###ეს ნოუთბუქი წმენდს მონაცემებს, შემდეგში ანალიტიკისთვის!

In [0]:
%sql
CREATE OR REPLACE TABLE getdata.calculated.cleaned_used_cars AS
SELECT 
    id,
    INITCAP(TRIM(manufacturer)) AS make,

    --MODEL-ის გაწმენდა: პირველი 50 სიმბოლო + ზედმეტი სიმბოლოების მოცილება
    INITCAP(
        TRIM(
            REGEXP_REPLACE(
                SUBSTRING(model, 1, 50),  -- პირველი 50 სიმბოლო
                '[*]+',  -- ასტერისკების მოცილება
                ''
            )
        )
    ) AS model,
    CAST(year AS INT) AS model_year,
    CAST(price AS INT) AS price,
    CAST(odometer AS INT) AS mileage,
    
    --NULL handling
    INITCAP(COALESCE(condition, 'Unknown')) AS condition,
    INITCAP(COALESCE(cylinders, 'Unknown')) AS cylinders,
    INITCAP(COALESCE(fuel, 'Other')) AS fuel_type,
    INITCAP(COALESCE(title_status, 'Unknown')) AS title_status,
    INITCAP(COALESCE(transmission, 'Other')) AS transmission,
    INITCAP(COALESCE(drive, 'Unknown')) AS drive_train,
    INITCAP(COALESCE(type, 'Other')) AS body_type,
    UPPER(COALESCE(state, 'Unknown')) AS state, 
    
    --  გარბენის სეგმენტაცია
    CASE 
        WHEN odometer < 50000 THEN 'Low'
        WHEN odometer BETWEEN 50000 AND 120000 THEN 'Moderate'
        WHEN odometer BETWEEN 120001 AND 200000 THEN 'High'
        ELSE 'Very High'
    END AS mileage_group,
    
    -- მანქანის ასაკის სეგმენტი
    CASE 
        WHEN year >= 2020 THEN 'New (2020+)'
        WHEN year BETWEEN 2015 AND 2019 THEN 'Recent'
        WHEN year BETWEEN 2010 AND 2014 THEN 'Mid-Age'
        WHEN year BETWEEN 2000 AND 2009 THEN 'Older'
        ELSE 'Classic'
    END AS vehicle_age_group,

     -- მანქანის ფასის სეგმენტი   
    CASE 
        WHEN price < 5000 THEN 'Budget'
        WHEN price BETWEEN 5000 AND 15000 THEN 'Economy'
        WHEN price BETWEEN 15001 AND 30000 THEN 'Mid-Range'
        WHEN price BETWEEN 30001 AND 60000 THEN 'Premium'
        ELSE 'Luxury'
    END AS price_range,
    
    -- თარიღი
    posting_date

FROM getdata.raw.craigslist_used_cars

--  გამწმენდი ფილტრები
WHERE price BETWEEN 500 AND 150000           -- რეალისტური ფასები
  AND odometer BETWEEN 1000 AND 400000       -- რეალისტური გარბენი
  AND year BETWEEN 1990 AND 2024             -- თანამედროვე მანქანები

  AND manufacturer IS NOT NULL               -- აუცილებელი ველები
  AND model IS NOT NULL
  AND price IS NOT NULL
  AND odometer IS NOT NULL;

In [0]:
%sql
select * from getdata.calculated.cleaned_used_cars limit 100;